In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# The Goal
In this notebook, I will be looking at different datasets that we have and try ro see which features from each stage in the sloar flare event -> CME-> solar wind -> geomagnetic storm relates to the occurrence of geomagnetic storm itself. It will help us to give a sense which features mught be useful for our model in the future, and which would just introduce noice without meaningful signal and thus can be ommites entirely.

In [2]:
solar_flare_events=pd.read_parquet('../data/clean-data/solar-flare-events.parquet')
omni_solar_wind_parameters=pd.read_parquet('../data/omni-solar-wind-parameters/omni_solar_wind_parameters.parquet')
donki_space_weather_events=pd.read_parquet('../data/donki-space-weather-events/donki_events.parquet')
space_weather_indices=pd.read_parquet('../data/space-weather-indices/space_weather_indices.parquet')


# Thoughts on what I can check

check correlation of `is_storm` with upstream variables from each dataset (skip `solar-wind/` — only starts 2026, too short).

reminder: skip the geomagnetic *response* columns, they leak the target — `ap_*`, `kp_*`, `kp_sum`, `cp`, `c9`, `storm_level`, and OMNI `kp_index`/`ap_index_nt`/`dst_index_nt`/`ae_index_nt`/`al_index_nt`/`au_index_nt`/`pc_n_index`.

**omni-solar-wind-parameters** (main feature source, hourly → aggregate to daily, use min/max not just mean):

- `bz_gsm_nt` — THE driver; southward (negative) Bz opens the magnetosphere. use `min(Bz)` over the day or `Bs = max(0, -Bz)`
- `electric_field_mvpm` — speed × southward field; the storm driver in operational models
- `flow_speed_kms` — fast wind = more energy in
- `b_magnitude_vector_nt` — more total field = more to reconnect
- `flow_pressure_npa` — ram pressure compresses the magnetosphere
- `proton_density_cm3` — feeds pressure + ring current (weaker)
- `by_gsm_nt` — Russell–McPherron seasonal projection onto Bz (weak)
- engineered coupling functions (Newell, VBs, Akasofu ε) — combine speed + field; usually beat any single raw column

**space-weather-indices** (target lives here, only F10.7 / sunspot are usable):

- `f107_obs` / `f107_adj` (+ 81-day means), `sunspot_number` — solar-cycle phase; sets the base rate of storms, weak same-day trigger

**solar-flare-events** (GOES, 2017+; lag features ~1–3 days, it's the CME not the X-rays that drives the storm):

- `peak_flux_wm2` / `goes_class` — M/X flares flag an active Sun / CME likelihood (noisy proxy)
- flare duration (`end_time - start_time`) — long-duration events tie to CMEs

**donki-space-weather-events** (2010+; lag ~1–3 days):

- `cme_speed_kms` — dominant control on big storms
- `cme_half_angle_deg` — halo (wide) CMEs are the Earth-directed ones
- `cme_latitude` / `cme_longitude` / `source_location` — is it aimed at Earth
- event counts by type (CME/HSS/SEP) — overall activity level